# Eigendecomposition and SVD

**Goal:** Implement power iteration to find the top eigenpair, validate against `torch.linalg.eigh`, explore SVD-based low-rank approximation, and connect SVD to PCA.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

**Note:** `torch.linalg.eigh` and `torch.linalg.eigvals` are not implemented on MPS as of PyTorch 2.x.  
Those specific calls run on a `.cpu()` copy with a brief comment; all other tensors remain on `device`.

In [1]:
import sys
from pathlib import Path

import torch


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure

device = configure()  # reads config.toml -> device/seed/dtype (mps on Apple Silicon)
print("running on:", device)

running on: mps


## From Scratch: Power Iteration

Power iteration repeatedly multiplies a vector by the matrix and renormalises.  
It converges to the **eigenvector corresponding to the largest eigenvalue**.

In [2]:
def power_iteration(
    A: torch.Tensor,
    n_iter: int = 500,
    tol: float = 1e-6,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Find top eigenpair of a real symmetric matrix via power iteration.

    Args:
        A: Symmetric matrix of shape (n, n).
        n_iter: Maximum iterations.
        tol: Convergence tolerance on eigenvector change.

    Returns:
        (eigenvalue, eigenvector) — both on the same device as A.
    """
    n = A.shape[0]
    # Random init; normalise
    v = torch.randn(n, device=A.device, dtype=A.dtype)
    v = v / v.norm()

    for _ in range(n_iter):
        v_new = A @ v
        v_new = v_new / v_new.norm()
        if (v_new - v).norm() < tol or (v_new + v).norm() < tol:
            break
        v = v_new

    v = v_new
    # Ensure consistent sign convention (largest component positive)
    if v.abs().argmax().item() != v.argmax().item():
        v = -v
    lam = (v @ A @ v)  # Rayleigh quotient
    return lam, v


# Build a 6x6 symmetric positive-definite matrix
torch.manual_seed(0)
M = torch.randn(6, 6, device=device)
A_sym = M @ M.T  # symmetric PSD

lam_scratch, v_scratch = power_iteration(A_sym)
print("Top eigenvalue (scratch):", lam_scratch.item())
print("Top eigenvector (scratch):", v_scratch)

Top eigenvalue (scratch): 11.239511489868164
Top eigenvector (scratch): tensor([-0.5290, -0.0031,  0.7308,  0.3185,  0.1459, -0.2517], device='mps:0')


## Validation Against `torch.linalg.eigh`

`torch.linalg.eigh` is not implemented on MPS — we run it on a CPU copy.

In [3]:
# eigh not supported on MPS; run on cpu copy
A_sym_cpu = A_sym.cpu()
eigvals_cpu, eigvecs_cpu = torch.linalg.eigh(A_sym_cpu)
# eigh returns eigenvalues in ascending order; top is last
top_val_ref = eigvals_cpu[-1].to(device)
top_vec_ref = eigvecs_cpu[:, -1].to(device)

print("Top eigenvalue (eigh ref):", top_val_ref.item())

# Align sign convention
if torch.dot(v_scratch, top_vec_ref) < 0:
    top_vec_ref = -top_vec_ref

assert torch.allclose(lam_scratch, top_val_ref, atol=1e-2), (
    f"Eigenvalue mismatch: {lam_scratch.item()} vs {top_val_ref.item()}"
)
assert torch.allclose(v_scratch, top_vec_ref, atol=1e-2), "Eigenvector mismatch"
print("Power iteration matches torch.linalg.eigh (atol=1e-2) ✓")
print("All eigenvalues (ascending):", eigvals_cpu)

Top eigenvalue (eigh ref): 11.23951244354248
Power iteration matches torch.linalg.eigh (atol=1e-2) ✓
All eigenvalues (ascending): tensor([ 0.4403,  1.4629,  2.9745,  4.2051,  6.5386, 11.2395])


## Singular Value Decomposition (SVD)

For any matrix A of shape (m, n): `A = U @ diag(S) @ Vh`  
- U: orthonormal columns (left singular vectors)  
- S: non-negative singular values (descending)  
- Vh: orthonormal rows (right singular vectors)

SVD auto-falls back to CPU on MPS with a warning — we explicitly use `.cpu()` to keep it clean.

In [4]:
torch.manual_seed(42)
m, n = 8, 5
A = torch.randn(m, n, device=device)

# Compute SVD on cpu copy (avoids MPS fallback warning on MPS; fine on cuda/cpu)
A_cpu = A.cpu()
U, S, Vh = torch.linalg.svd(A_cpu, full_matrices=False)  # economy SVD
U = U.to(device)
S = S.to(device)
Vh = Vh.to(device)

print("A shape:", A.shape)
print("U shape:", U.shape)
print("S shape:", S.shape)
print("Vh shape:", Vh.shape)
print("\nSingular values (descending):", S)

# Verify reconstruction: A = U @ diag(S) @ Vh
A_reconstructed = U @ torch.diag(S) @ Vh
assert torch.allclose(A, A_reconstructed, atol=1e-5), "SVD reconstruction failed!"
print("\nFull SVD reconstruction error:", (A - A_reconstructed).norm().item(), "(should be ~0) ✓")

A shape: torch.Size([8, 5])
U shape: torch.Size([8, 5])
S shape: torch.Size([5])
Vh shape: torch.Size([5, 5])

Singular values (descending): tensor([4.2192, 3.2009, 1.9859, 1.6286, 0.3456], device='mps:0')



Full SVD reconstruction error: 3.7427500956255244e-06 (should be ~0) ✓


## Low-Rank Approximation via Truncated SVD

The best rank-k approximation (Eckart-Young theorem): `A_k = U_k @ diag(S_k) @ Vh_k`  
Reconstruction error decreases as k grows.

In [5]:
print(f"{'k':>3}  {'Frobenius error':>18}  {'% variance captured':>20}")
print("-" * 46)
total_var = (S ** 2).sum()

for k in range(1, n + 1):
    A_k = U[:, :k] @ torch.diag(S[:k]) @ Vh[:k, :]
    error = (A - A_k).norm().item()
    var_captured = ((S[:k] ** 2).sum() / total_var * 100).item()
    print(f"{k:>3}  {error:>18.6f}  {var_captured:>19.2f}%")

# Assert: rank-n reconstruction matches full matrix
A_full = U @ torch.diag(S) @ Vh
assert torch.allclose(A, A_full, atol=1e-5), "Full rank reconstruction failed!"
print("\nRank-n reconstruction matches original ✓")

  k     Frobenius error   % variance captured
----------------------------------------------


  1            4.118413                51.21%
  2            2.591476                80.68%


  3            1.664875                92.03%


  4            0.345642                99.66%
  5            0.000004               100.00%

Rank-n reconstruction matches original ✓


## Relationship: SVD Singular Values vs Eigenvalues of `A^T A`

Singular values of A are the **square roots of eigenvalues of A^T A**.

In [6]:
# Compute eigenvalues of A^T A on CPU (eigh not on MPS)
AtA_cpu = (A.T @ A).cpu()
eigvals_AtA, _ = torch.linalg.eigh(AtA_cpu)
eigvals_AtA = eigvals_AtA.to(device)

# Sort descending to match SVD order
eigvals_AtA_desc = eigvals_AtA.flip(0)
S_squared = S ** 2

# Clamp small negatives from floating-point noise before sqrt
sing_from_eig = eigvals_AtA_desc.clamp(min=0).sqrt()

print("S from SVD:           ", S.cpu().tolist())
print("sqrt(eig(A^T A)):     ", sing_from_eig.cpu().tolist())
assert torch.allclose(S, sing_from_eig, atol=1e-4), "S != sqrt(eig(A^T A))!"
print("\nS == sqrt(eigenvalues of A^T A) ✓")

S from SVD:            [4.21916389465332, 3.2008705139160156, 1.9859347343444824, 1.6286004781723022, 0.3456418514251709]
sqrt(eig(A^T A)):      [4.219164848327637, 3.2008719444274902, 1.985934853553772, 1.6286007165908813, 0.34564343094825745]

S == sqrt(eigenvalues of A^T A) ✓


## PCA via SVD

PCA on a centered data matrix X (shape n_samples x d_features):  
1. Center: `X_c = X - X.mean(dim=0)`  
2. SVD: `U, S, Vh = svd(X_c)` — right singular vectors (rows of Vh) are **principal components**  
3. Project: `coords = X_c @ Vh.T`

In [7]:
torch.manual_seed(0)
n_samples, d = 100, 4
# Synthetic correlated data: 2 strong directions
true_components = torch.randn(2, d, device=device)
true_components = true_components / true_components.norm(dim=1, keepdim=True)
strengths = torch.tensor([5.0, 2.0], device=device)
latent = torch.randn(n_samples, 2, device=device)
X_data = latent * strengths @ true_components + 0.1 * torch.randn(n_samples, d, device=device)

# Center
X_c = X_data - X_data.mean(dim=0)

# SVD on cpu copy
X_c_cpu = X_c.cpu()
U_pca, S_pca, Vh_pca = torch.linalg.svd(X_c_cpu, full_matrices=False)
U_pca = U_pca.to(device)
S_pca = S_pca.to(device)
Vh_pca = Vh_pca.to(device)

# Project to top-2 components
coords_2d = X_c @ Vh_pca[:2].T  # (100, 2)
var_ratio = (S_pca[:2] ** 2).sum() / (S_pca ** 2).sum() * 100
print("Data shape:", X_data.shape)
print("Top-2 PCA coords shape:", coords_2d.shape)
print(f"Variance captured by top-2 components: {var_ratio:.1f}%")
print("\nSVD of centered data = PCA. Right singular vectors (Vh rows) are principal components.")

Data shape: torch.Size([100, 4])
Top-2 PCA coords shape: torch.Size([100, 2])
Variance captured by top-2 components: 99.9%

SVD of centered data = PCA. Right singular vectors (Vh rows) are principal components.


## Takeaways

- **Eigendecomposition:** `A = V Lambda V^-1`; for symmetric matrices `A = Q Lambda Q^T` (orthonormal Q). Eigenvalues describe scaling; eigenvectors describe directions preserved by the map.
- **Power iteration** converges to the top eigenpair — useful when only the largest eigenvalue is needed (e.g. spectral norm, condition number estimate).
- **SVD:** `A = U @ diag(S) @ Vh` works for *any* matrix (rectangular, rank-deficient). Singular values are always non-negative and sorted.
- **Low-rank approx:** truncating to top-k singular values gives the best rank-k approximation (Eckart-Young). Compression improves as k grows, with rank-n being exact.
- **SVD and PCA are equivalent:** SVD of a centered data matrix directly yields principal components (rows of Vh) and explained variance (S^2 / total).
- **MPS note:** `torch.linalg.eigh` and `torch.linalg.eigvals` are not yet supported on MPS — always compute on a `.cpu()` copy in notebooks running on Apple Silicon.